In [1]:
import open3d as o3d
import numpy as np
import os
import sys
import matplotlib.pyplot as plt

# Only needed for tutorial
sys.path.append('..')

import open3d.examples as o3dtut
# Change to True if you want to interact with the visualization windows
o3dtut.interactive = not 'CI' in os.environ

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 1. Preparing input data

In [2]:
pcd = o3d.io.read_point_cloud('DemoCloud/fragment.ply')
print(np.asarray(pcd.points).shape)
o3d.visualization.draw_geometries([pcd],
                                 zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])
               
# Using Voxel
voxel_down_pcd = pcd.voxel_down_sample(voxel_size = 0.05)
print(np.asarray(voxel_down_pcd.points).shape)
o3d.visualization.draw_geometries([voxel_down_pcd],
                                 zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])    
# Using Uniform Down Sample
down_pcd = pcd.uniform_down_sample(every_k_points=5)
print(np.asarray(down_pcd.points).shape)
o3d.visualization.draw_geometries([down_pcd],
                                 zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])  

(196133, 3)
(4718, 3)
(39227, 3)


## 2. Select down sample
- The following helper function uses select_by_index, which takes a binary mask to output only the selected points. The selected points and the non-selected points are visualized.

In [3]:
def display_inlier_outlier(cloud, ind):
    inlier_cloud = cloud.select_by_index(ind)
    outlier_cloud = cloud.select_by_index(ind, invert=True)

    print("Showing outliers (red) and inliers (gray): ")
    outlier_cloud.paint_uniform_color([1, 0, 0])
    inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud],
                                 zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])

## 3. Statistical outlier removal
- statistical_outlier_removal removes points that are further away from their neighbors compared to the average for the point cloud. It takes two input parameters:

- nb_neighbors, which specifies how many neighbors are taken into account in order to calculate the average distance for a given point.

- std_ratio, which allows setting the threshold level based on the standard deviation of the average distances across the point cloud. The lower this number the more aggressive the filter will be.

In [4]:
print("Statistical oulier removal")
cl, ind = voxel_down_pcd.remove_statistical_outlier(nb_neighbors=20,
                                                    std_ratio=2.0)
display_inlier_outlier(voxel_down_pcd, ind)

Statistical oulier removal
Showing outliers (red) and inliers (gray): 


## 4. Radius outlier removal
- radius_outlier_removal removes points that have few neighbors in a given sphere around them. Two parameters can be used to tune the filter to your data:

- nb_points, which lets you pick the minimum amount of points that the sphere should contain.

- radius, which defines the radius of the sphere that will be used for counting the neighbors.

In [5]:
print("Radius oulier removal")
cl, ind = voxel_down_pcd.remove_radius_outlier(nb_points=16, radius=0.05)
display_inlier_outlier(voxel_down_pcd, ind)

Radius oulier removal
Showing outliers (red) and inliers (gray): 
